### Install environment

In [ ]:
import importlib.util
if importlib.util.find_spec("flashrank") is None:
    !pip install vllm==0.10.0
    !pip install triton==3.2.0
    !pip install flashrank langchain langchain-community langchain_google_genai openai faiss-cpu langchain_huggingface crawl4ai unidecode pymupdf4llm google-genai
    !pip uninstall -y openai
    !pip install openai==1.90.0
else:
    print("All libs aldready installed")

In [ ]:
!wget -q -O ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -o ngrok.zip
!mv ngrok /usr/local/bin/ngrok
!chmod +x /usr/local/bin/ngrok
!ps aux | grep ngrok

In [ ]:
import os
# Fixed, do not change
os.environ["GLOO_SOCKET_NAME"] = "eth0"
os.environ["NCCL_SOCKET_NAME"] = "eth0"
os.environ["VLLM_HOST_IP"] = "127.0.0.1" # Internal ip for data communicate between VLLM components
os.environ["VLLM_USE_V1"] = "0" # T4 have compute capacity of 7.5, it need at least 8.0 to use V1

##### Download package from server

In [ ]:
DOMAIN = "http://127.0.0.1:8000"
BASE_PATH = ""

In [ ]:
import requests
import io
import tarfile
import shutil
def unpack_folder(data: bytes, path: str):
    if os.path.exists(path): # Remove old code
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_file(data: bytes, path: str):
    if os.path.exists(path):
        os.remove(path)
    with open(path, 'wb') as file:
        file.write(data)
def unpack_list(*names: str):
    if DOMAIN == "http://127.0.0.1:8000": return
    for name in names:
        if "." in name:
            url = f"{DOMAIN}/package/{name}"
        else:
            url = f"{DOMAIN}/package/{name}"
        data = requests.get(url).content
        if "." in name:
            unpack_file(data, name)
        else:
            unpack_folder(data, name)
unpack_list(
    "retriever", "server", "static.pkl", "worker.env", "data_collect"
)

### Config

Load environment variable (api keys)

In [ ]:
from dotenv import load_dotenv
load_dotenv(f"{BASE_PATH}worker.env")

In [ ]:
# import shutil, os
# if os.path.exists(f"{BASE_PATH}/data_logs"):
#     shutil.rmtree(f"{BASE_PATH}/data_logs")

Setup ngrok

In [ ]:
NGROK_PORT = 8002
if DOMAIN != "http://127.0.0.1:8000":
    import subprocess
    subprocess.Popen(["ngrok", "config", "add-authtoken", os.getenv("NGROK_TOKEN_1", "")])
    subprocess.Popen(["ngrok", "http", str(NGROK_PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

Login hugging face

In [ ]:
from huggingface_hub import login
login(os.getenv("HUGGING_FACE_TOKEN"))

##### Config

In [ ]:
from retriever import DataRetrieverPage, WebsearchConfig, RagConfig, SplitterConfig, PageRankerConfig, MergeNeighborConfig, MergeTableConfig, CmdLogger, SourceFormat
from server import *
from data_collect import DataCollector
from typing import AsyncGenerator, NotRequired, Protocol
from typing import Callable, AsyncGenerator
from openai import AsyncOpenAI, OpenAI
from google import genai
from google.genai import types
import os
import pickle
import json
import asyncio
import enum
import traceback

class CallType(enum.Enum):
    ROUTER = "router"
    RERANK = "page_rerank"
    READER = "reader"

class ModelProtocol(Protocol):
    async def __call__(self, call_type: CallType, instruction: str, prefix: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]: ...

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
# Retriever config
search_config = WebsearchConfig(
    page_timeout=15,
    file_timeout=15,
    concurrent_page=10,
    concurrent_download=8
)
rag_config = RagConfig(
    embedding_name="intfloat/multilingual-e5-base",
    device="cuda"
)
splitter_config = SplitterConfig(
    tokenizer_name=MODEL_ID,
    chunk_size=512,
    chunk_overlap=0,
    # device="cuda"
)
page_rerank_config = PageRankerConfig(
    use_llm=True,
    call=None
)
table_merge_config = MergeTableConfig(
    k_max_previous=3,
    k_max_next=3
)
neighbor_config = MergeNeighborConfig(
    k_previous_chunks=1,
    k_next_chunks=1
)
# Sampling Params
PAGE_RERANKER_PARAMS = GenerationParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=4096
)
ROUTER_PARAMS = GenerationParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=1024
)
MODELS: list[ModelInfo] = [
    {
        "name": "GPT 4o mini",
        "id": "gpt-4o-mini"
    },
    {
        "name": "Gemini 2.5 flash",
        "id": "gemini-2.5-flash"
    }
]
CLIENT_INFO: WorkerServerInfo = {
    "name": "Test Reader only",
    "domain": "http://127.0.0.1:8002", # Auto change when run with ngrok
    "models": MODELS
}

##### Template, instruction, prefix

In [ ]:
PAGE_RERANKER_INSTRUCTION =  "Bạn là hệ thống xếp hạng lại (reranker) kết quả tìm kiếm để trả lời truy vấn của người dùng (Query)."
PAGE_RERANKER_PREFIX = """Hướng dẫn:  
1. Trích xuất các thực thể chính trong truy vấn (tổ chức, khoa, viện, bộ môn, chủ đề, ...).  
2. Với mỗi ứng viên:  
   - Xác định thực thể chính mà nó đề cập.  
   - Gán mức độ ưu tiên: "High", "Medium" hoặc "Low" (xem quy tắc bên dưới).  
   - Viết ngắn gọn lý do.  
3. Gom nhóm các ứng viên theo mức ưu tiên.  
4. Trong mỗi nhóm, so sánh trực tiếp các ứng viên để quyết định thứ tự cuối cùng.  
5. Rà soát: Đảm bảo rằng High > Medium > Low toàn cục. Điều chỉnh nếu cần.  
6. Chỉ trả lời bằng JSON hợp lệ:
  - Suy nghĩ và điền vào "intermediate"
  - Kết luận và đưa ra "output"
7. Không được đưa thêm văn bản ngoài JSON.  

Quy tắc ưu tiên:  
- High: Trang nói rõ về đúng thực thể trong truy vấn.  
- Medium: Trang nói về thực thể liên quan chặt chẽ (cùng trường nhưng khác khoa/bộ môn) đến truy vấn.  
- Low: Trang chung chung về trường đại học hoặc không liên quan đến truy vấn.  

Input format: 
Query:  
"<user query>" 
Candidates:
[
  {"index": 1, "title": <string>, "description": <string>},
  {"index": 2, "title": <string>, "description": <string>},
  ...
]

Output format:
{
  "intermediate": [
    {"index": 1, "title": <string>, "entity_match": <bool>, "priority": "High|Medium|Low", "opinion": <string>},
    {"index": 2, "title": <string>, "entity_match": <bool>, "priority": "High|Medium|Low", "opinion": <string>},
    ...
  ],
  "output": [
    {"index": 2, "rank": 1, "score": <float 0.0–1.0>, "title": <string>, "reason": <string>},
    {"index": 1, "rank": 2, "score": <float 0.0–1.0>, "title": <string>, "reason": <string>},
    ...
  ]
}

Lưu ý: "intermediate" và "output" phải bao gồm tất cả "Candidates" từ "Input", kể cả khi có mức độ ưu tiên thấp. 
"""
PAGE_RERANKER_TEMPLATE = """Query: 
"{query}"
Candidates:
{pages}"""
ROUTER_INSTRUCTION = "Bạn là chuyên gia tạo từ khóa tìm kiếm thông minh. Nhiệm vụ: phân tích câu hỏi và tạo từ khóa giúp tìm được thông tin CĂN BẢN để LLM có thể suy luận ra câu trả lời."
ROUTER_PREFIX = """CHIẾN LƯỢC TÌM KIẾM:

1. **Phân tích ý định câu hỏi**: Xác định thông tin gì cần thiết để trả lời
2. **Xác định nguồn tìm kiếm**: Tìm kiếm từ local vector database hoặc tìm kiếm trên web. Vector DB có chứa thông tin về điểm chuẩn các năm, thông tin chung của trường, học phí và thông tin tuyển sinh. Các câu hỏi ngoài phạm vi của Vector DB sẽ tìm kiếm trên web.
2. **Tìm nguồn thông tin gốc**: Thay vì tìm trực tiếp câu trả lời, tìm dữ liệu để suy luận
3. **Tối ưu từ khóa**: 
   - Dùng thuật ngữ chính thức, tên đầy đủ
   - Xoá từ filler (xin, làm ơn, cho mình) nhưng giữ từ thể hiện ý định.
   - Ưu tiên giữ lại đối tượng (ví dụ như: tổ chức, trường, ngành, ...) → nhưng loại bỏ chi tiết con để tạo phạm vi tìm kiếm bao quát hơn.
   - Nếu input không có năm → thêm `2025`. Nếu có năm → giữ nguyên.
4. **Trả lời đúng định dạng**: Định dạng câu trả lời như sau:
- Nếu sử dụng web search: {"type_search": "web_search", "key_word": ["từ khóa 1", "từ khóa 2", ...]}
- Nếu sử dụng local vector DB: {"type_search": "vector_db", "key_word": [{"school_id": "tên trường 1", "section": "section1"},{"school_id": "tên trường 2", "section": "section2"},...]}. Lưu ý: section là 1 trong 4 mục: "thong_tin_chung", "hoc_phi", "diem_chuan", "tuyen_sinh", trong đó "thong_tin_chung" bao gồm thông tin chung của trường như tên, địa chỉ, liên hệ..., "hoc_phi" bao gồm thông tin học phí của trường, "diem_chuan" bao gồm điểm chuẩn các năm, "tuyen_sinh" bao gồm các ngành đào tạo của trường, thông tin tuyển sinh của truờng.

VÍ DỤ THÔNG MINH:

Câu hỏi: "Số tiến sĩ trong viện trí tuệ nhân tạo UET là bao nhiêu?"
→ Không có trong vector DB nên cần tìm kiếm trên web
→ Cần: Danh sách giảng viên để đếm tiến sĩ
→ Từ khóa: "danh sách giảng viên viện trí tuệ nhân tạo UET"
→ Trả về: {"type_search": "web_search", "key_word": ["danh sách giảng viên viện trí tuệ nhân tạo UET"]}

Câu hỏi: "Điểm chuẩn UET 2024?"
→ Vector DB có thông tin điểm chuẩn các năm nên tìm trong DB
→ Cần: Điểm chuẩn của Đại học Công nghệ - Đại học Quốc gia Hà Nội (UET)
→ Trả về: {"type_search": "vector_db", "key_word": [{"school_id": "UET", "section": "diem_chuan"}]}

Câu hỏi: "Điểm chuẩn ngành CNTT Bách Khoa 2025?"
→ Vector DB có thông tin điểm chuẩn các năm nên tìm trong DB
→ Cần: Điểm chuẩn của Đại học Bách Khoa (HUST)
→ Trả về: {"type_search": "vector_db", "key_word": [{"school_id": "HUST", "section": "diem_chuan"}]}

Câu hỏi: "Học phí ngành Luật kinh doanh VNU-LS như thế nào?"
→ Vector DB có thông tin học phí nên tìm trong DB
→ Cần: Học phí của Trường Đại học Luật - Đại học Quốc gia Hà Nội (LS)
→ Trả về: {"type_search": "vector_db", "key_word": [{"school_id": "LS", "section": "hoc_phi"}]}

Câu hỏi: "Chương trình đào tạo ngành Ngôn ngữ Anh Trường Đại học Ngoại ngữ - Đại học Quốc gia Hà Nội?"
→ Vector DB không có thông tin cụ thể về chương trình đào tạo của từng ngành nên cần tìm kiếm trên web
→ Cần: Toàn bộ học phần chương trình đào tạo ngành Ngôn ngữ Anh của Trường Đại học Ngoại ngữ - Đại học Quốc gia Hà Nội (ULIS)
→ Từ khóa: "toàn bộ học phần chương trình đào tạo ngành Ngôn ngữ Anh ULIS" hoặc "chương trình đào tạo ngành Ngôn ngữ Anh ULIS"
→ Trả về (chỉ 1 trong 2 tùy trường hợp): {"type_search": "web_search", "key_word": ["toàn bộ học phần chương trình đào tạo ngành Ngôn ngữ Anh ULIS"]} hoặc {"type_search": "web_search", "key_word": ["chương trình đào tạo ngành Ngôn ngữ Anh ULIS"]}

Câu hỏi: "Địa chỉ của UEB và Học viện Công nghệ Bưu chính Viễn thông?"
→ Vector DB có thông tin chung như địa chỉ nên tìm trong DB
→ Trả về: {"type_search": "vector_db", "key_word": [{"school_id": "UEB", "section": "thong_tin_chung"},{"school_id": "PTIT", "section": "thong_tin_chung"}]}

NGUYÊN TẮC:
- Các câu hỏi ngoài phạm vi vector DB (thông tin chung (gồm tên trường, giới thiệu, mã trường, địa chỉ, thông tin liên hệ như điện thoại, web ...), điểm chuẩn các năm, học phí, thông tin tuyển sinh) thì tìm kiếm trên web
- Nếu câu hỏi cần tìm thông tin mới nhất, ví dụ trong câu hỏi có từ khóa như "mới nhất", "cập nhật",... thì ưu tiên tìm kiếm trên web
- Tìm "danh sách", "bảng", "chương trình" thay vì câu hỏi trực tiếp

Chỉ trả về từ khóa, không giải thích."""
ROUTER_TEMPLATE = """Câu hỏi: {question}"""
READER_INSTRUCTION = """"Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất. Nếu được cung cấp link nguồn thì thêm vào phần cuối câu trả lời, nếu không được cung cấp thì không thêm."""
READER_UNTRAINED_INSTRUCTION = """Bạn là một AI tư vấn tuyển sinh đại học chuyên nghiệp. Hãy trả lời các câu hỏi một cách chính xác, hữu ích và thân thiện. Có thể sử dụng những thông tin được cung cấp để đưa ra câu trả lời hoặc lời khuyên tốt nhất. Nếu được cung cấp link nguồn thì thêm vào phần cuối câu trả lời, nếu không được cung cấp thì không thêm.
Nhiệm vụ của bạn: Trả lời câu hỏi theo hướng dẫn sau đây, phải tuân thủ định dạng Markdown chuẩn.
Hiện tại là năm 2025, sử dụng thông tin mới nhất có thể.
Hãy trả lời **chỉ bằng Markdown** theo form chuẩn dưới đây. 
Không thêm giải thích ngoài lề, không in JSON, không in XML.
"""
READER_UNTRAINED_PREFIX = """### FORM TRẢ LỜI (Markdown):

### {TIÊU ĐỀ NGẮN}
**Tóm tắt:**  
- {Ý chính 1}  
- {Ý chính 2}  
- {Ý chính 3 (nếu có)}  

{BẢNG CHÍNH nếu có số liệu, theo đúng intent}  
| Cột 1 | Cột 2 | Cột 3 | (Cột 4 nếu cần) |
|-------|-------|-------|-----------------|
| ...   | ...   | ...   | ...             |

**Nguồn:** [Tên nguồn 1](URL1), [Tên nguồn 2](URL2)  
**Lưu ý:** {ràng buộc hoặc phạm vi áp dụng}  

**Bạn có thể hỏi thêm:** {2–4 gợi ý follow-up câu hỏi liên quan}

---

# Quy tắc bắt buộc
1. **Tiêu đề** luôn bắt đầu bằng `###`, chứa loại thông tin + năm + trường/ngành.  
2. **Tóm tắt** luôn 2–3 bullet, có con số chính yếu (điểm/học phí/số lượng/địa chỉ).  
3. **Bảng chính** chỉ hiển thị nếu có dữ liệu số hoặc danh sách.  
4. **Nguồn**: luôn có ≥1 link; nếu không tìm thấy thì ghi “(chưa tìm thấy nguồn đáng tin)”.  
5. **Lưu ý**: ghi rõ phạm vi (năm, phương thức, chương trình, cơ sở,...).  
6. **Follow-up**: luôn gợi ý 2–4 câu hỏi liên quan.  
7. Nếu dữ liệu thiếu → ghi rõ trong “Lưu ý” hoặc “Nguồn”.  
8. Không bao giờ trả lời ngoài form này.
9. Trong ví dụ câu hỏi điểm chuẩn, tôi chỉ liệt kê 4 ngành ví dụ. Trong câu trả lời thực tế, mỗi trường liệt kê ra cho tôi khoảng 15-20 ngành(phù hợp với trường đó). Tương tự với câu hỏi chỉ tiêu tuyển sinh các ngành.
10. Nếu không tìm thấy đủ dữ liệu đáng tin để xây dựng đầy đủ theo form, chỉ trả lời: "Tôi không tìm thấy thông tin phù hợp.". Sau đó bắt buộc gợi ý người dùng hỏi lại (ví dụ: thêm năm, trường, ngành, phương thức...), hoặc đưa ra 2–4 câu hỏi liên quan.
---

# Ví dụ input → output.

**Ví dụ 1 — Input (câu hỏi):**  
“Điểm chuẩn ngành CNTT UET 2024 là bao nhiêu?”

**Output (Markdown):**
### Điểm chuẩn 2024 — Ngành CNTT (UET, THPT)  
**Tóm tắt:**  
- Điểm chuẩn ngành CNTT UET năm 2024 là **27.80**.  
- Áp dụng cho phương thức xét tuyển THPT.  
- Mã ngành: CN1.  

| Ngành               | Phương thức | Mã ngành | Điểm chuẩn |
|---------------------|-------------|----------|------------|
| Công nghệ thông tin | THPT        | CN1      | 27.80      |

**Nguồn:** [UET Công khai 2024](https://...)  
**Lưu ý:** Điểm chuẩn thay đổi theo phương thức khác (học bạ, ĐGNL).  

**Bạn có thể hỏi thêm:** học phí ngành CNTT, chỉ tiêu 2025, tổ hợp xét tuyển.  

---

**Ví dụ 2 — Input (câu hỏi):**  
“Đội ngũ giảng viên của UET như thế nào?”

**Output (Markdown):**
### Đội ngũ giảng viên — UET (2024)  

**Tóm tắt:**  
- Tổng số giảng viên cơ hữu: ~200.  
- Khoảng **15% là Giáo sư/Phó Giáo sư**.  
- Hơn **80% có bằng Tiến sĩ hoặc Thạc sĩ**.  

| Học hàm / Học vị | Số lượng | Tỉ lệ   |
|------------------|----------|---------|
| GS/PGS           | 30       | 15%     |
| TS/ThS           | 170      | 85%     |
| Khác             | 5        | ~2%     |

**Nguồn:** [UET Công khai đội ngũ 2024](https://...)  
**Lưu ý:** Số liệu mang tính tham khảo, có thể thay đổi theo từng năm học.  

**Bạn có thể hỏi thêm:**  
- Tỉ lệ giảng viên/sinh viên tại UET là bao nhiêu?  
- Các giảng viên tiêu biểu trong ngành CNTT?  
- Nhóm nghiên cứu mạnh nào đang hoạt động tại UET?

---

**Ví dụ 3 — Input (câu hỏi):**
“Điểm chuẩn theo phương thức THPT của Đại học Bách khoa (HUST) năm 2024?”

**Output (Markdown):**
### Điểm chuẩn Đại học Bách khoa (HUST) 2024 — Phương thức THPT  
**Tóm tắt:**  
- Điểm chuẩn Đại học Bách khoa năm 2024 phân bổ từ **24.00** đến **29.50** tùy ngành.
- Ngành có điểm chuẩn cao nhất là **Công nghệ thông tin** với **29.50** điểm.
- Ngành có điểm chuẩn thấp nhất là **Kỹ thuật cơ khí** với **24.00** điểm.

| Ngành               | Phương thức | Mã ngành | Điểm chuẩn |
|---------------------|-------------|----------|------------|
| Công nghệ thông tin | THPT        | IT1      | 29.50      |
| Kỹ thuật cơ khí     | THPT        | ME1      | 24.00      |
| Kỹ thuật điện       | THPT        | EE1      | 26.50      |
| Kỹ thuật xây dựng   | THPT        | CE1      | 25.00      |


**Nguồn:** [Điểm chuẩn HUST 2024](https://...)  
**Lưu ý:** Điểm chuẩn thay đổi theo phương thức khác (học bạ, ĐGNL).  

**Bạn có thể hỏi thêm:** học phí ngành CNTT, chỉ tiêu 2025, tổ hợp xét tuyển.  

--- 
"""
READER_TEMPLATE = """Hiện tại là năm 2025, sử dụng thông tin mới nhất có thể. 
Nếu không tìm thấy đủ dữ liệu đáng tin để xây dựng đầy đủ theo form, chỉ trả lời: "Tôi không tìm thấy thông tin phù hợp.".
Thông tin tham khảo:
{context}
Câu hỏi:
{question}"""

### Utility class

##### Data Retriever

In [ ]:
class StaticDBSearch:
    """Search in static db"""
    def __init__(self) -> None:
        with open(f"{BASE_PATH}static.pkl", 'rb') as file:
            self.all_docs = pickle.load(file)
    def _filter_docs(self, school_id: str, section: str) -> list:
        school_docs = [doc for doc in self.all_docs if doc.metadata.get("school_id") == school_id]
        if school_docs:
            filtered_docs = [doc for doc in school_docs if doc.metadata.get("section") == section]
        else:
            filtered_docs = []
        return filtered_docs
    def __call__(self, keywords: list[dict]) -> list[WebSource]:
        results: list[WebSource] = []
        try:
            for kw in keywords:
                school_id = kw.get("school_id")
                section = kw.get("section")
                if school_id and section:
                    docs = self._filter_docs(school_id, section)
                    title = f"Tìm trường ĐH-CĐ - Cốc Cốc ({school_id})"
                    combined_content = "\n\n".join([doc.page_content for doc in docs])
                    description = combined_content[:100] + "..." if len(combined_content) > 100 else combined_content
                    web_source: WebSource = {
                        "title": title,
                        "description": description,
                        "url": "https://hoctap.coccoc.com/tim-truong-dh-cd",
                        "text": combined_content,
                        "files": []
                    }
                    results.append(web_source)
        except:
            traceback.print_exc()
        finally:            
            return results
    def from_websources(self, sources: list[WebSource]) -> list[RagSource]:
        results: list[RagSource] = []
        for source in sources:
            rag_source: RagSource = {
                "title": source["title"],
                "url": source["url"],
                "text": source["text"]
            }
            results.append(rag_source)
        return results

In [ ]:
class WebSearch:
    """Search in web"""
    def __init__(self, llm_call: ModelProtocol) -> None:
        page_rerank_config.call = self._llm_reranker
        self.web_search = DataRetrieverPage(
            search_config=search_config,
            rag_config=rag_config,
            splitter_config=splitter_config,
            page_rerank_config=page_rerank_config,
            merge_neighbor_config=neighbor_config,
            merge_table_config=table_merge_config
        )
        self.llm_call = llm_call
    def _construct_reranker_prompt(self, query: str, data: list[tuple[str, str]]) -> str:
        candidates = [{
            "index": index+1, 
            "title": item[0],
            "description": item[1],
        } for index, item in enumerate(data)]
        candidates = "[" + ",\n".join([json.dumps(item, ensure_ascii=False) for item in candidates]) + "]"
        return PAGE_RERANKER_TEMPLATE.format(query=query, pages=candidates)
    async def start(self):
        """Initialize websearch"""
        await self.web_search.start()
    async def _llm_reranker(self, query: str, data: list[tuple[str, str]]) -> list[float]:
        if len(data) == 0: return []
        text = ""    
        scores = [0.0 for _ in data]
        prompt = self._construct_reranker_prompt(query, data)
        async for chunk in await self.llm_call(
            call_type=CallType.RERANK, 
            instruction=PAGE_RERANKER_INSTRUCTION, 
            prefix=PAGE_RERANKER_PREFIX,
            prompt=prompt, 
            params=PAGE_RERANKER_PARAMS
        ):
            text += chunk
        try:
            result = json.loads(text)
            if "output" in result:
                for item in result["output"]:
                    index = int(item["index"]) - 1
                    scores[index] = float(item["score"])
            else:
                # Fallback if model not provide intermediate step
                for item in result:
                    index = int(item["index"]) - 1
                    scores[index] = float(item["score"])
        except:
            print(text)
            traceback.print_exc()
        return scores
    async def websearch(self, query: str, k_pages: int, domain_restrict: bool, engine: Literal["google", "brave"]):
        return await self.web_search.websearch(query, k_pages, domain_restrict, engine, include_pdf=False, include_image=False)
    async def from_websources(self, web_sources: list[WebSource], rerank_query: str, rag_query: str, params: GenerationParams) -> list[RagSource]:
        k_docs = params.get("k_docs", 0)
        if k_docs == 0:
            return []
        else:
            rag_sources = await self.web_search.rag_retrieve(
                web_sources, rerank_query, rag_query, k_docs,
                rerank_chunk=True, merge_table=True, merge_neighbor=False
            )
            return rag_sources

In [ ]:
class DataRetriverRouter:
    def __init__(self, model: ModelProtocol) -> None:
        self.static = StaticDBSearch()
        self.web = WebSearch(model)
        self.model = model
    async def start(self):
        await self.web.start()
    async def _llm_route(self, question: str):
        prompt = ROUTER_TEMPLATE.format(question=question)
        generator = await self.model(
            call_type=CallType.ROUTER, 
            instruction=ROUTER_INSTRUCTION,
            prefix=ROUTER_PREFIX, 
            prompt=prompt, 
            params=ROUTER_PARAMS
        )
        text = ""
        async for chunk in generator:
            text += chunk
        return text
    async def router(self, question: str):
        """Sử dụng router để quyết định nguồn"""
        try:
            # Sử dụng router để quyết định hướng tìm kiếm
            text = await self._llm_route(question)
            print(text)
            search_strategy: dict = json.loads(text)
            search_type = search_strategy.get("type_search")
            keywords: list[str | dict[str, str]] = search_strategy.get("key_word", [])
            return {"type_search": search_type, "key_words": keywords}
        except Exception as e:
            return {"type_search": "web_search", "key_words": [question]}
    async def __call__(self, question: str, params: GenerationParams) -> tuple[str, list[WebSource], list[RagSource]]:
        """
        Retrive data with user question.\n
        Return web query, list of `WebSource` and list of `RagSource`.
        """
        k_pages = params.get("k_pages", 0)
        k_docs = params.get("k_docs", 0)
        domain_restrict = params.get("domain_restric", False)
        print(f"[WS] k_pages: {k_pages} | k_docs {k_docs} | domain_restrict: {domain_restrict}")
        if k_pages == 0 or k_docs == 0:
            return question, [], []
        else:
            strategy = await self.router(question)
            if strategy["type_search"] == "vector_db":
                web_sources = self.static(strategy["key_words"])
                rag_sources = self.static.from_websources(web_sources)
                web_query = strategy["key_words"]
            if strategy["type_search"] == "vector_db" and (len(web_sources) == 0 or web_sources[0]["text"] == ""):
                # Fallback to web search
                strategy["type_search"] = "web_search"
                strategy["key_words"] = [question]
            if strategy["type_search"] != "vector_db":
                # Multi query is a little bit too complicated
                web_query = ";".join([str(item) for item in strategy["key_words"]])
                web_sources = []
                rag_sources = []
                for query in strategy["key_words"]:
                    web_sources_ = await self.web.websearch(query, k_pages, domain_restrict, params.get("engine_type", "brave"))
                    web_sources.extend(web_sources_)
                    rag_sources.extend(await self.web.from_websources(web_sources_, question, query, params))
        return web_query, web_sources, rag_sources
    async def from_websources(self, query: str, sources: list[WebSource], params: GenerationParams, rag_query: str | None = None) -> list[RagSource]:
        """For multi-turn."""
        dynamic_sources: list[WebSource] = []
        rag_sources: list[RagSource] = []
        for source in sources:
            if source["url"] == "https://hoctap.coccoc.com/tim-truong-dh-cd":
                rag_sources.extend(self.static.from_websources([source]))
            else:
                dynamic_sources.append(source)
        if rag_query is None:
            rag_query = str(params.get("kwargs", {}).get("rag_query", query)) # Try to get rag_query, fallback to query
        rag_queries = rag_query.split(";") if ";" in rag_query else [rag_query] * len(dynamic_sources)
        for source, rag_query_ in zip(dynamic_sources, rag_queries):
            rag_sources.extend(await self.web.from_websources([source], query, rag_query_, params))
        return rag_sources

##### Client to call model

In [ ]:
class APIModel:
    def __init__(self) -> None:
        self.gpt_client = AsyncOpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))
        self.gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
        self.current_model: Literal["gpt-4o-mini", "gemini-2.5-flash"] = "gpt-4o-mini"
    async def call(self, call_type: CallType, instruction: str, prefix: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        print(f"[API] {call_type} | Instruction length: {len(instruction)} | Prompt length: {len(prompt)} | Prefix length: {len(prefix)} | kwargs: {params.get('kwargs')}")
        if "gpt" in self.current_model:
            stream = await self.gpt_client.chat.completions.create(
                model="gpt-4o-mini", 
                messages=[
                    {"role": "system", "content": instruction},
                    {"role": "user", "content": prefix+prompt}
                ],
                max_tokens=params.get("max_tokens", 4096),
                temperature=params.get("temperature", 0.7),
                top_p=params.get("top_p", 0.9),
                presence_penalty=params.get("presence_penalty", 0.1),
                frequency_penalty=params.get("frequency_penalty", 0.0),
                stream=True
            )
            total_text = ""
            async for event in stream:
                chunk = event.choices[0].delta.content
                if chunk is not None:
                    total_text += chunk
                    yield chunk
        else:
            gemini_config = types.GenerateContentConfig(
                temperature=params.get("temperature", 0.8),
                top_p=params.get("top_p", 0.9),
                top_k=params.get("top_k", 16),
                max_output_tokens=params.get("max_tokens", 2048)
            )
            stream = await self.gemini_client.aio.models.generate_content_stream(
                model="gemini-2.5-flash",
                contents=[
                    types.Content(
                        role="user",
                        parts=[types.Part.from_text(text=instruction)]
                    ),
                    types.Content(
                        role="user",
                        parts=[types.Part.from_text(text=prefix+prompt)]
                    )
                ],
                config=gemini_config,
            )
            async for chunk in stream:
                text = chunk.text
                if text != None:
                    yield text
    async def __call__(self, call_type: CallType, instruction: str, prefix: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        return self.call(call_type, instruction, prefix, prompt, params)

##### Pipeline

In [ ]:
class CustomQA:
    def __init__(self, model_protocol: ModelProtocol) -> None:
        self.logger = CmdLogger("QA")
        self.retriever = DataRetriverRouter(model_protocol)
        self.llm_call = model_protocol
    async def start(self):
        await self.retriever.start()
    async def inference(self, prompt: str, request: WorkerChatRequest) -> AsyncGenerator[str, None]:
        text = ""
        async for chunk in await self.llm_call(
            call_type=CallType.READER, 
            instruction=READER_INSTRUCTION, 
            prefix="",
            prompt=prompt, 
            params=request["params"]
        ):
            text += chunk
            yield chunk
    async def pre_inference(
        self,
        question: str,
        stream_id: str,
        params: GenerationParams
    ) -> tuple[str, ModelPreOutput]:
        web_query, web_sources, rag_sources = await self.retriever(
            question, 
            params
        )
        context = SourceFormat()(rag_sources)
        prompt = READER_TEMPLATE.format(context=context, question=question)
        self.logger.start()
        pre_output: ModelPreOutput = {
            "model_id": MODEL_ID,
            "generation_params": params,
            "web_sources": web_sources,
            "rag_sources": rag_sources,
            "extra_data": {
                "web_query": web_query,
            },
            "result_url": stream_id,
            "user_intent": "",
            "user_keywords": [],
            "user_summary": ""
        }
        return prompt, pre_output

### Final

VLLMM Engine

In [ ]:
api_model = APIModel()

Websearch pipeline

In [ ]:
ws_pipeline = CustomQA(api_model)
await ws_pipeline.start()
REQUEST_STORAGE: dict[str, tuple[str, WorkerChatRequest, ModelPreOutput]] = {}

async def pre_inference_function(request: WorkerChatRequest) -> ModelPreOutput:
    api_model.current_model = request["model_id"]
    prompt, pre_output = await ws_pipeline.pre_inference(
        request["text"],
        request["stream_id"],
        request["params"]
    ) 
    REQUEST_STORAGE[request["stream_id"]] = (prompt, request, pre_output)
    DataCollector.pre_output(pre_output)
    return pre_output

async def inference_function(stream_id: str) -> AsyncGenerator[str, None]:
    prompt, request, pre_output = REQUEST_STORAGE.pop(stream_id)
    api_model.current_model = request["model_id"]
    generator = ws_pipeline.inference(prompt, request)
    total = ""
    try:
        async for chunk in generator:
            total += chunk
            yield chunk
    finally:
    # Store chat data when finish
        model_output: ModelOutput = {
            **pre_output,
            "answer_state": "successfully",
            "bot_summary": total[:50],
            "bot_keywords": [],
            "text": total
        }
        data: WorkerStoreChatData = {
            "forward_kwargs": request["forward_kwargs"],
            "model_output": model_output
        }
        DataCollector.model_output(data)
        DataCollector.backup(f"{BASE_PATH}data_logs")
        await app.state.store_chat(data)

Connect to server

In [ ]:
app = construct_app(
    server_domain=DOMAIN,
    info=CLIENT_INFO,
    pre_inference=pre_inference_function,
    inferece=inference_function,
    init_tasks=[],
    shutdown_tasks=[],
    is_local=True
)
# CORS policy
from fastapi.middleware.cors import CORSMiddleware
origins = [
    "http://127.0.0.1:8000",
    DOMAIN
]
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)
import nest_asyncio
import uvicorn
nest_asyncio.apply()
uvicorn.run(app, port=NGROK_PORT)

###### Note 
If the instruction is too long -> freeze. Seem like vLLM cache instruction, but we still don't know why it only freeze after some requests. (Seem like cache problem, use llm serve would cause it)


In [ ]:
!curl http://localhost:4040